# Appendix A — Construction of Lebesgue Measure

Companion notebook to Appendix A of *Advanced Calculus for Machine Learning*.
Every numerical claim of the appendix is recomputed here and checked with an
assertion.

Most of the appendix is about sets that no computation reaches: arbitrary
coverings, all test sets, a set chosen with the axiom of choice. What can be
computed is computed exactly, with Python fractions, or with PyTorch on
grids. Random samples are used in three cells, which say so.

In [1]:
import math
import itertools
from fractions import Fraction as Fr
import torch

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)
T = torch.tensor

2.15.0+cu132


## 1. Counting grid points (Lemma A.5)

For a box $R = \prod_i[a_i, b_i)$ the number of points of
$\frac1N\mathbb Z^n$ in $R$ is
$\prod_i(\lceil Nb_i\rceil - \lceil Na_i\rceil)$, and divided by $N^n$ it
tends to the volume. Counting is exact for unions, which gives the
inequality for finite coverings.

In [2]:
# boxes are given in thousandths, so that every comparison is between integers
def count(box, N):
    # number of points of (1/N) Z^n in the half-open box, by the formula with ceilings
    c = 1
    for a, b in box:
        c *= math.ceil(Fr(N * b, 1000)) - math.ceil(Fr(N * a, 1000))
    return c


def vol(box):
    v = Fr(1)
    for a, b in box:
        v *= Fr(b - a, 1000)
    return v


def inside(box, M, N):
    # M: integer points m, standing for the points m / N
    ok = torch.ones(M.shape[0], dtype=torch.bool)
    for i, (a, b) in enumerate(box):
        ok &= (1000 * M[:, i] >= a * N) & (1000 * M[:, i] < b * N)
    return ok


def grid(N, lo, hi):
    # the points of (1/N) Z^2 in [lo, hi)^2, as integer numerators
    m = torch.arange(N * lo, N * hi)
    X, Y = torch.meshgrid(m, m, indexing="ij")
    return torch.stack([X.reshape(-1), Y.reshape(-1)], dim=1)


R = [(137, 2710), (-1050, 620)]
for N in (10, 100, 1000):
    brute = int(inside(R, grid(N, -2, 4), N).sum())
    assert brute == count(R, N)
    err = abs(Fr(count(R, N), N ** 2) - vol(R))
    # each factor differs from N(b - a) by less than 1
    assert err <= (Fr(2710 - 137, 1000) + Fr(620 + 1050, 1000)) / N + Fr(1, N ** 2)
    print(f"N = {N:5d}: {count(R, N):8d} grid points, count / N^2 = {count(R, N) / N ** 2:.6f}, volume = {float(vol(R)):.6f}")

# a finite covering: the three boxes of Figure A.2 cover R = [0, 4) x [0, 2.5)
R0 = [(0, 4000), (0, 2500)]
cover = [[(-450, 2000), (-300, 2900)], [(1700, 4600), (1100, 3100)], [(1200, 4450), (-500, 1400)]]
N = 200
P = grid(N, -1, 5)
in_R = inside(R0, P, N)
in_cover = torch.zeros_like(in_R)
for B in cover:
    in_cover |= inside(B, P, N)
assert bool((in_cover | ~in_R).all())                       # every grid point of R is covered
assert count(R0, N) <= sum(count(B, N) for B in cover)
assert vol(R0) <= sum(vol(B) for B in cover)
print(f"|R| = {float(vol(R0)):.3f} <= sum of the |R_k| = {float(sum(vol(B) for B in cover)):.3f}")

# disjoint boxes inside R: the other inequality
parts = [[(0, 1500), (0, 2500)], [(1500, 4000), (0, 1000)], [(2000, 3500), (1200, 2400)]]
masks = [inside(B, P, N) for B in parts]
assert all(not bool((A & B).any()) for A, B in itertools.combinations(masks, 2))
assert sum(count(B, N) for B in parts) <= count(R0, N) and sum(vol(B) for B in parts) <= vol(R0)

N =    10:      442 grid points, count / N^2 = 4.420000, volume = 4.296910
N =   100:    42919 grid points, count / N^2 = 4.291900, volume = 4.296910


N =  1000:  4296910 grid points, count / N^2 = 4.296910, volume = 4.296910
|R| = 10.000 <= sum of the |R_k| = 19.815


## 2. A covering of the rational numbers (Example A.4, Figure A.1)

The rationals of $[0, 1]$ are listed as
$0, 1, \frac12, \frac13, \frac23, \frac14, \frac34, \dots$ and the $k$th
receives an interval of length $\epsilon2^{-k}$. Only finitely many can be
listed here: all those with denominator at most 60. Lengths are exact
fractions.

In [3]:
def rationals(max_den):
    seen, out = set(), [Fr(0), Fr(1)]
    seen.update(out)
    for q in range(2, max_den + 1):
        for p in range(1, q):
            x = Fr(p, q)
            if x not in seen:
                seen.add(x)
                out.append(x)
    return out


def merge(intervals):
    # union of intervals as a sorted list of disjoint intervals
    merged = []
    for a, b in sorted(intervals):
        if merged and a <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], b)
        else:
            merged.append([a, b])
    return merged


eps = Fr(1, 2)
qs = rationals(60)
assert qs[:7] == [Fr(0), Fr(1), Fr(1, 2), Fr(1, 3), Fr(2, 3), Fr(1, 4), Fr(3, 4)]
ivs = [(q - eps / 2 ** (k + 1), q + eps / 2 ** (k + 1)) for k, q in enumerate(qs, start=1)]
total = sum(b - a for a, b in ivs)
assert total == eps * (1 - Fr(1, 2 ** len(qs))) < eps
U = merge(ivs)
U01 = [(max(a, Fr(0)), min(b, Fr(1))) for a, b in U]
lam_U = sum(b - a for a, b in U01)
assert lam_U < total
print(f"{len(qs)} rationals listed; total length of the intervals = {float(total):.6f} < eps = {float(eps)}")
print(f"measure of their union inside [0, 1] = {float(lam_U):.6f}, in {len(U)} disjoint intervals")

# every point of the grid j/60 is one of the listed rationals: a grid sees U everywhere
assert all(Fr(j, 60) in set(qs) for j in range(61))

# random points see it with frequency close to its measure (random samples)
x = torch.rand(200_000)
lo, hi = T([float(a) for a, _ in U01]), T([float(b) for _, b in U01])
idx = torch.searchsorted(lo, x, right=True) - 1
hit = (idx >= 0) & (x < hi[idx.clamp(min=0)])
print(f"fraction of 200000 uniform points that fall in the union: {hit.double().mean().item():.4f}")
assert abs(hit.double().mean().item() - float(lam_U)) < 0.005

1103 rationals listed; total length of the intervals = 0.500000 < eps = 0.5
measure of their union inside [0, 1] = 0.312500, in 787 disjoint intervals


fraction of 200000 uniform points that fall in the union: 0.3121


### The indicator of this open set is not Riemann integrable (Example A.15)

On any subdivision of $[0, 1]$ into $N\leq60$ equal intervals, every
interval contains a listed rational, so every upper step function is at
least $1$ everywhere. The lower sums stay below the measure of the union.

In [4]:
listed = set(qs)


def contained(a, b):
    return any(u <= a and b <= v for u, v in U)


for N in (10, 30, 60):
    cells_ = [(Fr(j, N), Fr(j + 1, N)) for j in range(N)]
    upper = sum((b - a) for a, b in cells_ if a in listed)       # the left end is a listed rational: sup = 1
    lower = sum((b - a) for a, b in cells_ if contained(a, b))   # cells inside the union: inf = 1
    assert upper == 1 and lower <= lam_U
    print(f"N = {N}: upper sum = {upper}, lower sum = {float(lower):.4f}")
assert 1 - float(lam_U) > 0.5

N = 10: upper sum = 1, lower sum = 0.1000
N = 30: upper sum = 1, lower sum = 0.1333
N = 60: upper sum = 1, lower sum = 0.2000


## 3. Carathéodory's test on finite sets (Definition A.7, Exercise A.2)

On a finite set every subset can be tested against every other. First the
outer measure of Exercise A.2, for which only $\emptyset$ and $X$ are
measurable; then the outer measure of Proposition A.2 for the blocks of a
partition, for which the measurable sets are the unions of blocks.

In [5]:
def subsets(X):
    return [frozenset(c) for r in range(len(X) + 1) for c in itertools.combinations(X, r)]


def is_outer(mu, X):
    S = subsets(X)
    if mu(frozenset()) != 0:
        return False
    mono = all(mu(A) <= mu(B) for A in S for B in S if A <= B)
    sub = all(mu(A | B) <= mu(A) + mu(B) for A in S for B in S)
    return mono and sub


def measurable(mu, X):
    S = subsets(X)
    return [A for A in S if all(mu(E) == mu(E & A) + mu(E - A) for E in S)]


X3 = (1, 2, 3)
mu_star = lambda A: 0 if len(A) == 0 else (2 if len(A) == 3 else 1)
assert is_outer(mu_star, X3)
assert mu_star(frozenset({1, 2})) < mu_star(frozenset({1})) + mu_star(frozenset({2}))
M = measurable(mu_star, X3)
assert set(M) == {frozenset(), frozenset(X3)}
print("Exercise A.2: the measurable sets are", [sorted(A) for A in M])

# the outer measure of the coverings by the blocks of a partition, with prices rho
X6 = (0, 1, 2, 3, 4, 5)
blocks = {frozenset({0, 1}): Fr(1, 2), frozenset({2, 3, 4}): Fr(1, 3), frozenset({5}): Fr(1, 6)}


def mu_blocks(A):
    # cheapest covering by blocks: brute force over the families of blocks
    best = None
    for r in range(len(blocks) + 1):
        for fam in itertools.combinations(blocks, r):
            if A <= frozenset().union(*fam):
                cost = sum(blocks[b] for b in fam)
                best = cost if best is None else min(best, cost)
    return best


assert is_outer(mu_blocks, X6)
M6 = set(measurable(mu_blocks, X6))
unions = {frozenset().union(*fam) for r in range(4) for fam in itertools.combinations(blocks, r)}
assert M6 == unions and len(M6) == 8
assert all(mu_blocks(A | B) == mu_blocks(A) + mu_blocks(B) for A in M6 for B in M6 if not (A & B))
print("blocks of a partition: the measurable sets are the 8 unions of blocks, and mu* is additive on them")

Exercise A.2: the measurable sets are [[], [1, 2, 3]]


blocks of a partition: the measurable sets are the 8 unions of blocks, and mu* is additive on them


## 4. Half-spaces cut boxes exactly (Lemma A.9)

In [6]:
gen = torch.Generator().manual_seed(1)
for _ in range(1000):
    a = torch.rand(3, generator=gen) * 2 - 1
    b = a + torch.rand(3, generator=gen)
    i = int(torch.randint(0, 3, (1,), generator=gen))
    c = float(torch.rand(1, generator=gen) * 4 - 2)
    cp = min(max(c, float(a[i])), float(b[i]))
    left, right = (b - a).clone(), (b - a).clone()
    left[i], right[i] = cp - a[i], b[i] - cp
    assert torch.allclose(left.prod() + right.prod(), (b - a).prod())
print("|R ∩ H| + |R \\ H| = |R| for 1000 random boxes and half-spaces")

|R ∩ H| + |R \ H| = |R| for 1000 random boxes and half-spaces


## 5. Open sets outside, compact sets inside (Theorem A.12)

For a disc of radius $r$: the union of the closed dyadic squares of side
$2^{-k}$ contained in the disc is compact, the union of the squares that
meet it contains it, and the two areas squeeze $\pi r^2$.

In [7]:
r, centre = 0.83, T([0.11, -0.07])
prev_gap = None
for k in range(4, 10):
    h = 2.0 ** -k
    m = torch.arange(-2 ** k, 2 ** k)
    X, Y = torch.meshgrid(m * h, m * h, indexing="ij")
    lo = torch.stack([X.reshape(-1), Y.reshape(-1)], dim=1)          # lower left corners
    far = torch.maximum((lo - centre).abs(), (lo + h - centre).abs())  # farthest corner, coordinate by coordinate
    near = torch.clamp(centre, lo, lo + h)                             # nearest point of the closed square
    inner = (far.norm(dim=1) <= r).sum().item() * h * h
    outer = ((near - centre).norm(dim=1) <= r).sum().item() * h * h
    assert inner <= math.pi * r * r <= outer
    gap = outer - inner
    if prev_gap is not None:
        assert gap < 0.75 * prev_gap
    prev_gap = gap
    print(f"k = {k}: inner {inner:.6f} <= pi r^2 = {math.pi * r * r:.6f} <= outer {outer:.6f}   (gap {gap:.5f})")
assert prev_gap < 0.03

k = 4: inner 1.964844 <= pi r^2 = 2.164243 <= outer 2.386719   (gap 0.42188)
k = 5: inner 2.061523 <= pi r^2 = 2.164243 <= outer 2.270508   (gap 0.20898)
k = 6: inner 2.113770 <= pi r^2 = 2.164243 <= outer 2.217773   (gap 0.10400)
k = 7: inner 2.139099 <= pi r^2 = 2.164243 <= outer 2.191101   (gap 0.05200)
k = 8: inner 2.151566 <= pi r^2 = 2.164243 <= outer 2.177505   (gap 0.02594)
k = 9: inner 2.157810 <= pi r^2 = 2.164243 <= outer 2.170780   (gap 0.01297)


## 6. The Cantor set (Example A.14, Figure A.4) and a fat one (Exercise A.4)

In [8]:
def cantor(k):
    ivs = [(Fr(0), Fr(1))]
    for _ in range(k):
        ivs = [piece for a, b in ivs for piece in ((a, a + (b - a) / 3), (b - (b - a) / 3, b))]
    return ivs


for k in range(9):
    C = cantor(k)
    assert len(C) == 2 ** k and all(b - a == Fr(1, 3 ** k) for a, b in C)
    assert sum(b - a for a, b in C) == Fr(2, 3) ** k
print("C_k: 2^k intervals of length 3^-k, total length (2/3)^k:", [str(Fr(2, 3) ** k) for k in range(5)])

# the points x_eps = sum 2 eps_k 3^-k, for the 2^8 sequences of length 8 (followed by zeros)
k = 8
C8 = cantor(k)
lefts = {a for a, _ in C8}
pts = {}
for e in itertools.product((0, 1), repeat=k):
    pts[e] = sum(Fr(2 * d, 3 ** (j + 1)) for j, d in enumerate(e))
assert set(pts.values()) == lefts                      # the left ends of the intervals of C_8
for e, f in itertools.combinations(pts, 2):
    first = next(j for j in range(k) if e[j] != f[j]) + 1
    assert abs(pts[e] - pts[f]) >= Fr(1, 3 ** first)
print("the 256 sequences of length 8 give 256 distinct points, separated as the example says")

# counting boxes: 2^k intervals of length 3^-k, and the exponent ln 2 / ln 3
d = math.log(2) / math.log(3)
for k in (4, 8):
    assert abs(math.log(len(cantor(k))) / math.log(3 ** k) - d) < 1e-12
print(f"number of intervals = (1/length)^d with d = ln 2 / ln 3 = {d:.6f}")

# Exercise A.4: remove 2^(k-1) intervals of length 4^-k at step k
ivs = [(Fr(0), Fr(1))]
for k in range(1, 11):
    g = Fr(1, 4 ** k)
    assert all(b - a > g for a, b in ivs)
    ivs = [piece for a, b in ivs for piece in ((a, (a + b - g) / 2), ((a + b + g) / 2, b))]
    assert len(ivs) == 2 ** k
    assert all(b - a == Fr(1, 2 ** (k + 1)) + Fr(1, 2 ** (2 * k + 1)) for a, b in ivs)
    assert sum(b - a for a, b in ivs) == Fr(1, 2) + Fr(1, 2 ** (k + 1))
print("fat Cantor set: after k steps the length is 1/2 + 2^-(k+1); the limit set has measure 1/2")

C_k: 2^k intervals of length 3^-k, total length (2/3)^k: ['1', '2/3', '4/9', '8/27', '16/81']


the 256 sequences of length 8 give 256 distinct points, separated as the example says
number of intervals = (1/length)^d with d = ln 2 / ln 3 = 0.630930
fat Cantor set: after k steps the length is 1/2 + 2^-(k+1); the limit set has measure 1/2


## 7. Coin flips (Example A.20, Exercise A.7)

The binary digits of a point of $[0, 1)$. First exactly, with intervals;
then on uniform samples (random samples), and in the other direction, from
random bits to a uniform number.

In [9]:
digit = lambda x, k: int(x * 2 ** k) % 2          # for a fraction x in [0, 1)

k = 4
for m in range(2 ** k):
    e = [(m >> (k - j)) & 1 for j in range(1, k + 1)]
    assert m == sum(ej * 2 ** (k - j) for j, ej in enumerate(e, start=1))
    for x in (Fr(m, 2 ** k), Fr(2 * m + 1, 2 ** (k + 1)), Fr(m + 1, 2 ** k) - Fr(1, 10 ** 9)):
        assert [digit(x, j) for j in range(1, k + 1)] == e      # the interval [m 2^-k, (m+1) 2^-k)
# lambda{d_1 = 1, d_3 = 1}: the intervals of length 1/8 with digits 1?1
good = [m for m in range(8) if (m >> 2) & 1 and m & 1]
assert good == [5, 7] and Fr(len(good), 8) == Fr(1, 4)

# on samples
n = 400_000
x = torch.rand(n)
D = torch.stack([torch.floor(x * 2 ** j) % 2 for j in range(1, 7)], dim=1)
freq = D.mean(dim=0)
assert (freq - 0.5).abs().max() < 0.005
patterns = (D[:, 0] * 4 + D[:, 1] * 2 + D[:, 2]).long()
counts = torch.bincount(patterns, minlength=8).double() / n
assert (counts - 1 / 8).abs().max() < 0.003
assert torch.corrcoef(D.T).fill_diagonal_(0).abs().max() < 0.01
print("frequencies of the 8 patterns of the first three digits:", [round(c, 4) for c in counts.tolist()])

# from 40 random bits to a uniform number
bits = torch.randint(0, 2, (n, 40)).double()
u = (bits * 2.0 ** -torch.arange(1, 41)).sum(dim=1)
assert abs(u.mean() - 0.5) < 0.003 and abs(u.var() - 1 / 12) < 0.002
assert abs((u < 0.3).double().mean() - 0.3) < 0.004
print(f"number built from 40 fair bits: mean {u.mean().item():.4f}, variance {u.var().item():.4f} (1/12 = {1 / 12:.4f})")

# Exercise A.7 (1): no digit 7 among the first k decimal digits
for k in (1, 5, 10):
    exact = Fr(9, 10) ** k
    xs = torch.rand(n)
    no7 = torch.ones(n, dtype=torch.bool)
    for j in range(1, k + 1):
        no7 &= (torch.floor(xs * 10 ** j) % 10) != 7
    assert abs(no7.double().mean().item() - float(exact)) < 0.005
print("no digit 7 in the first k decimals: measure (9/10)^k, which tends to 0")

# Exercise A.7 (3): upper sums of the indicator of a subset of the Cantor set
assert [sum(b - a for a, b in cantor(k)) for k in (5, 10)] == [Fr(2, 3) ** 5, Fr(2, 3) ** 10]

frequencies of the 8 patterns of the first three digits: [0.1261, 0.1258, 0.1241, 0.1242, 0.1253, 0.1253, 0.125, 0.1242]


number built from 40 fair bits: mean 0.5008, variance 0.0833 (1/12 = 0.0833)
no digit 7 in the first k decimals: measure (9/10)^k, which tends to 0


## 8. Null sets in practice (the machine learning box)

* PyTorch gives the derivative of ReLU at $0$ the value $0$.
* The floating-point numbers of $[1, 2)$ are $2^{52}$ equally spaced
  points: a finite set.
* After a ReLU layer, an exact zero is not a null event. For a Gaussian
  input $x$ in $\mathbb R^3$ and the layer with rows
  $e_1, e_2, e_3, e_1 + e_2$, all activations vanish when the three
  coordinates of $x$ are negative, with probability $\frac18$, and the next
  pre-activation is then exactly $0$ (random samples).

In [10]:
z = torch.zeros(3, requires_grad=True)
torch.relu(z).sum().backward()
assert torch.equal(z.grad, torch.zeros(3))
print("derivative of ReLU at 0 according to PyTorch:", z.grad.tolist())

one = torch.ones(())
assert (torch.nextafter(one, 2 * one) - one).item() == 2.0 ** -52

W = T([[1.0, 0, 0], [0, 1, 0], [0, 0, 1], [1, 1, 0]])
w2 = T([0.7, -1.3, 0.4, 2.1])
x = torch.randn(400_000, 3)
z2 = torch.relu(x @ W.T) @ w2
zero = (z2 == 0)
assert torch.equal(zero, (x <= 0).all(dim=1))
print(f"fraction of inputs for which the next pre-activation is exactly 0: {zero.double().mean().item():.4f}  (1/8 = 0.125)")
assert abs(zero.double().mean().item() - 1 / 8) < 0.003

derivative of ReLU at 0 according to PyTorch: [0.0, 0.0, 0.0]
fraction of inputs for which the next pre-activation is exactly 0: 0.1253  (1/8 = 0.125)


## 9. Vitali's argument on a finite group (Theorem A.18)

The set $V$ of the theorem cannot be computed. Its bookkeeping can be
imitated on $\mathbb Z/12$, with the subgroup $H = \{0, 4, 8\}$ in the role
of the rational numbers: one point is chosen in each class, and the
translates of the chosen set by $H$ are disjoint and cover the group. Here
there is no contradiction, because $H$ is finite: $12 = 3\times4$. In the
theorem the same equation reads $1\leq\infty\times\lambda(V)\leq3$.

In [11]:
n, H = 12, (0, 4, 8)
classes = {frozenset((x + h) % n for h in H) for x in range(n)}
V = sorted(min(c) + 0 for c in classes)                 # one point in each class
translates = [frozenset((v + h) % n for v in V) for h in H]
assert all(not (A & B) for A, B in itertools.combinations(translates, 2))
assert frozenset().union(*translates) == frozenset(range(n))
assert n == len(H) * len(V)
print("classes:", sorted(sorted(c) for c in classes), " V =", V)

classes: [[0, 4, 8], [1, 5, 9], [2, 6, 10], [3, 7, 11]]  V = [0, 1, 2, 3]


## Summary

* The number of grid points in a box is given by a product of differences
  of ceilings; divided by $N^n$ it tends to the volume, and it settles
  finite coverings.
* The rationals with denominator at most 60 are covered by intervals of
  total length below $\epsilon$; a grid sees the covering everywhere,
  random points see it with frequency equal to its measure, and upper and
  lower sums of its indicator stay apart.
* On finite sets Carathéodory's test can be run in full: it keeps two sets
  in one example and the unions of blocks in the other.
* Dyadic squares inside and around a disc squeeze its area.
* The Cantor set: $2^k$ intervals of total length $(2/3)^k$; a variant
  keeps measure $\frac12$ and still contains no interval.
* Binary digits of a uniform number are fair independent coin flips.
* Exact zeros occur with positive probability after a ReLU layer.